# Notebook 1 — Validate the C-sieve output against an independent Python sieve

**Context.** The prime-gap pipeline has two halves:

1. A C program (`main-csv-static`, built from `app-c/main-csv-static.c`) that sieves primes up to `N` and writes `gaps.csv` + `records.csv`.
2. A Python analysis layer on top of those CSVs (Hardy–Littlewood / Wolf comparisons, figures, `summary.md`).

We want to rule out bugs in *either* half before trusting the presented results. This notebook attacks **half 1**:

- Run the C binary here in the notebook directory with a chosen `N`.
- Independently sieve primes in pure Python/numpy to the same `N`.
- Compare the histogram row-by-row and the records table row-by-row.
- Check integer identities: $\pi(N) - 1 = $ total gap count; largest gap matches; the `gap=1` row (if any) is only the (2→3) artefact.

Any mismatch will be localised to the exact `gap` or record entry. If everything matches, half 1 is trustworthy at this `N` and we can move on to validating half 2 in the next notebook.

**No `.py` files from the broken local analysis dir are imported** — this notebook is fully self-contained.

**Runtime.** Python numpy sieve for $N = 10^{8}$: ~100 MB RAM, ~5–15 s. C sieve is much faster.

In [1]:
import sys, os, time, math, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd

print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)

# --- Configuration -----------------------------------------------------------
N_LIMIT  = 10**8                                  # upper bound for the sieve
NB_DIR   = Path.cwd()
C_BINARY = NB_DIR / 'main-csv-static'             # binary co-located with the notebook
WORK_DIR = NB_DIR / 'run_N1e8'                    # C writes gaps.csv / records.csv here
# -----------------------------------------------------------------------------

WORK_DIR.mkdir(exist_ok=True)
print('notebook :', NB_DIR)
print('C binary :', C_BINARY, '| exists:', C_BINARY.exists(),
      '| executable:', os.access(C_BINARY, os.X_OK))
print('work dir :', WORK_DIR)
print('N limit  :', f'{N_LIMIT:,}')
assert C_BINARY.exists() and os.access(C_BINARY, os.X_OK), f'C binary not usable: {C_BINARY}'


python 3.11.10 | numpy 2.0.2 | pandas 2.2.3
notebook : /home/jovyan/work
C binary : /home/jovyan/work/main-csv-static | exists: True | executable: True
work dir : /home/jovyan/work/run_N1e8
N limit  : 100,000,000


## 1. Run the C sieve

`main-csv-static <N> [stride]`. `stride = 0` disables the gap-stream output (we only need `gaps.csv` + `records.csv` here).

In [2]:
t0 = time.perf_counter()
proc = subprocess.run(
    [str(C_BINARY), str(N_LIMIT), '0'],
    cwd=WORK_DIR,
    capture_output=True, text=True, check=False,
)
t1 = time.perf_counter()
print(f'C sieve finished in {t1 - t0:.2f} s  (returncode={proc.returncode})')
print('--- stdout (tail) ---')
print('\n'.join(proc.stdout.splitlines()[-25:]))
if proc.stderr.strip():
    print('--- stderr ---')
    print(proc.stderr)
assert proc.returncode == 0, 'C binary failed'

ref_gaps    = pd.read_csv(WORK_DIR / 'gaps.csv')
ref_records = pd.read_csv(WORK_DIR / 'records.csv')
print()
print('ref_gaps rows       :', len(ref_gaps))
print('ref_records rows    :', len(ref_records))
print('total gaps reported :', int(ref_gaps["count"].sum()))

C sieve finished in 0.02 s  (returncode=0)
--- stdout (tail) ---
New record gap: 72 at p=31397  (merit=6.954)
New record gap: 86 at p=155921  (merit=7.192)
New record gap: 96 at p=360653  (merit=7.503)
New record gap: 112 at p=370261  (merit=8.735)
New record gap: 114 at p=492113  (merit=8.698)
New record gap: 118 at p=1349533  (merit=8.360)
New record gap: 132 at p=1357201  (merit=9.348)
New record gap: 148 at p=2010733  (merit=10.197)
New record gap: 154 at p=4652353  (merit=10.031)
New record gap: 180 at p=17051707  (merit=10.810)
New record gap: 210 at p=20831323  (merit=12.461)
New record gap: 220 at p=47326693  (merit=12.449)

Summary:
  Range:               [2, 100000000]
  Number of primes:    5761455
  Number of gaps:      5761454
  Largest gap:         220
  Number of records:   25
  Running time:        0.02 s

Output files:
  gaps.csv     - histogram of all gaps
  records.csv  - record gaps

ref_gaps rows       : 97
ref_records rows    : 25
total gaps reported : 5761454


## 2. Independent Python sieve

Plain Eratosthenes on a numpy bool array. Small enough for `N` up to a few times $10^{8}$ on a laptop.

In [3]:
def sieve_primes(n: int) -> np.ndarray:
    """Return sorted np.int64 array of all primes p with 2 <= p <= n."""
    if n < 2:
        return np.array([], dtype=np.int64)
    is_prime = np.ones(n + 1, dtype=bool)
    is_prime[:2] = False
    is_prime[4::2] = False
    for i in range(3, int(math.isqrt(n)) + 1, 2):
        if is_prime[i]:
            is_prime[i*i::2*i] = False
    return np.flatnonzero(is_prime).astype(np.int64)

t0 = time.perf_counter()
primes = sieve_primes(N_LIMIT)
t1 = time.perf_counter()
print(f'Python sieve done in {t1 - t0:.2f} s')
print(f'pi({N_LIMIT:,}) = {len(primes):,}')
print(f'first 10 : {primes[:10].tolist()}')
print(f'last  5  : {primes[-5:].tolist()}')

Python sieve done in 0.41 s
pi(100,000,000) = 5,761,455
first 10 : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
last  5  : [99999931, 99999941, 99999959, 99999971, 99999989]


In [4]:
gaps = np.diff(primes)
max_gap = int(gaps.max())
hist = np.bincount(gaps, minlength=max_gap + 1)

my_gaps = pd.DataFrame({'gap': np.arange(max_gap + 1), 'count': hist})
my_gaps = my_gaps[my_gaps['count'] > 0].reset_index(drop=True)
print('my_gaps rows:', len(my_gaps))
print('total gaps  :', int(my_gaps["count"].sum()))
my_gaps.head(10)

my_gaps rows: 97
total gaps  : 5761454


,gap,count
0,1,1
1,2,440312
2,4,440257
3,6,768752
4,8,334180
5,10,430016
6,12,538382
7,14,293201
8,16,215804
9,18,384738


In [5]:
running_max = 0
rec_rows = []
for i, g in enumerate(gaps):
    g = int(g)
    if g > running_max:
        running_max = g
        p_start = int(primes[i])
        rec_rows.append({
            'gap':     g,
            'p_start': p_start,
            'ln_p':    math.log(p_start),
            'merit':   g / math.log(p_start),
        })
my_records = pd.DataFrame(rec_rows)
print('my_records rows:', len(my_records))
my_records

my_records rows: 25


,gap,p_start,ln_p,merit
0,1,2,0.693147,1.442695
1,2,3,1.098612,1.820478
2,4,7,1.945910,2.055593
3,6,23,3.135494,1.913574
4,8,89,4.488636,1.782278
5,14,113,4.727388,2.961466
6,18,523,6.259581,2.875592
7,20,887,6.787845,2.946443
8,22,1129,7.029088,3.129851
9,34,1327,7.190676,4.728345


## 3. Histogram comparison (strict, integer equality)

In [6]:
joined = my_gaps.merge(ref_gaps, on='gap', how='outer',
                       suffixes=('_mine', '_ref')).fillna(0)
joined['diff'] = joined['count_mine'].astype(int) - joined['count_ref'].astype(int)

mismatches = joined[joined['diff'] != 0]
print(f'histogram rows, mine : {len(my_gaps)}')
print(f'histogram rows, ref  : {len(ref_gaps)}')
print(f'rows with mismatch   : {len(mismatches)}')
if len(mismatches):
    print('\nFIRST 50 MISMATCHES:')
    display(mismatches.sort_values('gap').head(50))
else:
    print('  PERFECT MATCH on the histogram.')

histogram rows, mine : 97
histogram rows, ref  : 97
rows with mismatch   : 0
  PERFECT MATCH on the histogram.


## 4. Records comparison

In [7]:
print('my_records:')
print(my_records.to_string(index=False))
print()
print('ref_records (from C):')
print(ref_records.to_string(index=False))

# The C output typically contains a `gap=1` artefact for the 2->3 pair. Filter it.
ref_filt  = ref_records[ref_records['gap'] >= 2].reset_index(drop=True)
mine_filt = my_records[my_records['gap']  >= 2].reset_index(drop=True)

if len(ref_filt) != len(mine_filt):
    print(f'\nROW COUNT MISMATCH: ref has {len(ref_filt)}, mine has {len(mine_filt)}')
else:
    ok_gap = bool((ref_filt['gap'].values     == mine_filt['gap'].values).all())
    ok_p   = bool((ref_filt['p_start'].values == mine_filt['p_start'].values).all())
    ok_ln  = bool(np.allclose(ref_filt['ln_p'],  mine_filt['ln_p'],  rtol=1e-6, atol=1e-6))
    ok_m   = bool(np.allclose(ref_filt['merit'], mine_filt['merit'], rtol=1e-6, atol=1e-6))
    print('\nRecords equality (gap, p_start, ln_p, merit):', ok_gap, ok_p, ok_ln, ok_m)
    if not ok_ln or not ok_m:
        cmp = ref_filt.merge(mine_filt, on=['gap', 'p_start'], suffixes=('_ref', '_mine'))
        cmp['dln']    = cmp['ln_p_ref']  - cmp['ln_p_mine']
        cmp['dmerit'] = cmp['merit_ref'] - cmp['merit_mine']
        display(cmp[['gap','p_start','ln_p_ref','ln_p_mine','dln','merit_ref','merit_mine','dmerit']])

my_records:
 gap  p_start      ln_p     merit
   1        2  0.693147  1.442695
   2        3  1.098612  1.820478
   4        7  1.945910  2.055593
   6       23  3.135494  1.913574
   8       89  4.488636  1.782278
  14      113  4.727388  2.961466
  18      523  6.259581  2.875592
  20      887  6.787845  2.946443
  22     1129  7.029088  3.129851
  34     1327  7.190676  4.728345
  36     9551  9.164401  3.928244
  44    15683  9.660333  4.554709
  52    19609  9.883744  5.261164
  72    31397 10.354468  6.953520
  86   155921 11.957105  7.192377
  96   360653 12.795672  7.502537
 112   370261 12.821963  8.735012
 114   492113 13.106464  8.697998
 118  1349533 14.115269  8.359741
 132  1357201 14.120935  9.347823
 148  2010733 14.514010 10.197044
 154  4652353 15.352884 10.030689
 180 17051707 16.651761 10.809668
 210 20831323 16.851968 12.461452
 220 47326693 17.672585 12.448660

ref_records (from C):
 gap  p_start      ln_p     merit
   1        2  0.693147  1.442695
   2        3

## 5. Aggregate sanity checks

In [8]:
total_ref  = int(ref_gaps['count'].sum())
total_mine = int(my_gaps['count'].sum())
print(f'total gaps, mine  = {total_mine:,}')
print(f'total gaps, ref   = {total_ref:,}')
print(f'pi(N) mine        = {len(primes):,}')
print(f'pi(N) - 1 == total_gaps_mine : {len(primes) - 1 == total_mine}')
print(f'pi(N) - 1 == total_gaps_ref  : {len(primes) - 1 == total_ref}')
print()
print(f'largest gap, mine = {int(my_gaps["gap"].max())}')
print(f'largest gap, ref  = {int(ref_gaps["gap"].max())}')
print()
print('gap=1 in ref_gaps    :', ref_gaps[ref_gaps["gap"] == 1].to_dict('records'))
print('gap=1 in ref_records :', ref_records[ref_records["gap"] == 1].to_dict('records'))
print('mine: primes[0], primes[1] =', int(primes[0]), int(primes[1]),
      '| diff =', int(primes[1] - primes[0]))

total gaps, mine  = 5,761,454
total gaps, ref   = 5,761,454
pi(N) mine        = 5,761,455
pi(N) - 1 == total_gaps_mine : True
pi(N) - 1 == total_gaps_ref  : True

largest gap, mine = 220
largest gap, ref  = 220

gap=1 in ref_gaps    : [{'gap': 1, 'count': 1}]
gap=1 in ref_records : [{'gap': 1, 'p_start': 2, 'ln_p': 0.693147, 'merit': 1.442695}]
mine: primes[0], primes[1] = 2 3 | diff = 1


## 6. Verdict

Tick after the run:

- [ ] Histogram identical (section 3, zero mismatched rows).
- [ ] Records identical (section 4, four `True` flags).
- [ ] $\pi(N) - 1 = $ total gaps (both mine and ref).
- [ ] Largest gap agrees; `gap=1` only the (2,3) artefact.

All four green → the C sieve is correct at $N = 10^{8}$. Next notebook: attack half 2 — independently recompute the Hardy–Littlewood / Wolf predictions and check whether the `emp/Wolf` ratios in `summary.md` reflect real physics or a Python bug.

Any failure → paste the mismatching rows, we localise the C bug from there.